# 时间外配置预测上的扣费动作标签
只用2020–2022年度OOF预测，连续平滑与账户不跨年重置。固定2bp费用，训练参考策略禁用全部学习型开盘/恢复模型，保留确定性风险投影。执行与持有分支共享次日目标重置，标签到次日0949才成熟。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec,ACTION_FEATURES
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.tiered_action_labels import own_path_labels
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/tangent_router_labels_20261015';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
oof,oa=_source(root,plan['oof_source'],['analysis/oof_predictions','models/2020_71']);source,sa=_source(root,plan['price_source'],['inputs/frame','inputs/plan']);training,ta=_source(root,plan['training_source'],['inputs/daily_frame']);action,aa=_source(root,plan['action_source'],['inputs/plan','controller/source_recovery_selection'])
assert source['inputs/plan']['tariff']==plan['tariff']
pred=oof['analysis/oof_predictions'];model=oof['models/2020_71'];returns=training['inputs/daily_frame'].loc[pred.index,['soxl_forward_open_return','soxx_forward_open_return']];returns.columns=['soxl_return','soxx_return']
targets=_smooth_weights(pred.drop(columns=['trade_date','model_year']),spec=model.spec,candidate=model.candidate,asset_returns=returns)
frame=source['inputs/frame'].set_index('signal_date').reindex(pred.index).rename_axis('signal_date').reset_index();assert frame.trade_date.notna().all();assert pd.DatetimeIndex(frame.trade_date).equals(pd.DatetimeIndex(pred.trade_date))
for col in targets:frame[col]=targets[col].to_numpy()
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);controller={'frozen_open_action_value_model':None,'frozen_recovery_model':None};selection=action['controller/source_recovery_selection']
replay=replay_full_path(frame,models=None,plan=plan,slippage_bps=plan['training_slippage_bps'],controller=controller,recovery_spec=recovery,action_spec=spec,selection=selection)
assert replay['replay'].execute_open_target.all() and not replay['replay'].recovery_action.any()
book=native_tiered_stage1(frame,replay,provider_uri=provider,tariff=plan['tariff'],slippage_bps=plan['training_slippage_bps'],initial_cash=plan['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);book['targets']=replay['targets']
labels=own_path_labels(frame,book,controller=controller,recovery_spec=recovery,action_spec=spec,selection=selection,plan=plan,slippage_bps=plan['training_slippage_bps'],training_reference=True)
labels['decision_at']=pd.to_datetime(labels.decision_at).dt.tz_localize('America/New_York');labels['label_available_at']=pd.to_datetime(labels.label_available_at).dt.tz_localize('America/New_York')
labels['training_qualified']=labels.eligible&labels.label_available_at.lt(pd.Timestamp(plan['label_cutoff']))&np.isfinite(labels.tiered_advantage)&np.isfinite(labels[list(ACTION_FEATURES)]).all(axis=1)
assert (labels.loc[labels.training_qualified,'label_available_at']>labels.loc[labels.training_qualified,'decision_at']).all()
assert not labels.iloc[-1].training_qualified
rows=[]
for year,g in labels.groupby(labels.trade_date.dt.year):
 q=g[g.training_qualified];rows.append({'year':year,'reference_days':len(g),'eligible_days':int(g.eligible.sum()),'mature_qualified_rows':len(q),'positive_execute_fraction':float(q.tiered_advantage.gt(0).mean()),'mean_advantage_bps':float(1e4*q.tiered_advantage.mean()),'last_label_available_at':str(q.label_available_at.max())})
table=pd.DataFrame(rows);policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/native_reference','analysis/labels','analysis/targets']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_audits={'oof':oa,'prices':sa,'daily':ta,'action':aa},frame=frame,label_code=(root/'src/analysis/tiered_action_labels.py').read_text());rec.save_objects(artifact_path='analysis',native_reference=book,labels=labels,targets=targets,year_summary=table);rec.log_metrics(native_cases=1,qualified_action_rows=int(labels.training_qualified.sum()),registered_artifact_count=8)
summary={'experiment':card.name,'decision':'action_labels_ready_not_promoted','new_fits':0,'new_native_cases':1,'reference_sessions':len(frame),'qualified_action_rows':int(labels.training_qualified.sum()),'minimum_router_training_rows':spec.minimum_training_rows,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':plan['scope'],'independent_strategy_validation':False,'objective_achieved':False}
table.to_csv(card/'label_coverage.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/plan['oof_source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table.to_string(index=False))


# 时间外配置的动作标签：260条成熟样本，尚无路由收益验证

完成1个756日连续Qlib原生参考账户，0新模型拟合。使用2020/2021/2022的年度OOF配置预测，原目标EMA/保持规则跨年连续，参考账户现金、股数、风险状态与Tiered累计股数不跨年重置。固定10万美元、IBKR Pro Tiered投影、2bp滑点；仅训练参考账户禁用学习型开盘与恢复模型，保留确定性风险投影，避免用训练至2022的控制器在2020构造所谓时间外状态。

原生账户与独立结算逐日核对通过。标签重建再次复核风险状态、时钟、执行决定、股数和现金与参考路径一致。每个合格小额、非去风险且持仓合法的交易，从同一账户和月度费用状态分出执行与HOLD两支，分别扣成交费用，次日0947按相同参考目标重置，再比较净值；标签到次日0949才成熟。两支费用状态互不污染。标签是局部共同重置优势，不是未来整条自主策略的收益。

| 年度 | 参考日数 | 成熟可训练动作 | 执行优势为正比例 | 平均净优势bp |
|---|---:|---:|---:|---:|
| 2020 | 253 | 67 | 35.82% | +0.0115 |
| 2021 | 252 | 87 | 40.23% | +0.7915 |
| 2022 | 251 | 106 | 27.36% | -0.1604 |

共260条合格标签，严格成熟早于2023-01-01；其他日期和最后一日保留但不可训练，没有将缺失优势填为0。正类比例是历史动作分布，不是分类器预测准确率，也不能由平均优势直接推出统一HOLD规则。

原路由minimum_training_rows=200保持不变。全期260条满足最终拟合数量门，但2020–2021仅154条，无法按原门槛训练后独立检验2022。不得降低门槛、随机打乱交叉验证或将2022训练标签泄漏进其验证。即使之后做全期固定参数拟合和2023+完整回测，也仅能视为复用历史诊断，不能补足独立开发/前向证据。

训练参考账户不含学习型恢复，未来部署候选若保留恢复，其状态分布存在差异，必须在完整自主账户回测中检验，不能把局部标签效果当作完整策略改进。价格/派息及费用仍是既定历史研究口径，非真实券商结算认证。配置结构已在历史研究中选择，OOF仅证明模型训练时序，不证明研究过程独立。

决策action_labels_ready_not_promoted。下一项若继续，只允许沿用原逻辑回归特征/C/概率阈值作一次全期拟合，固定0/2/5bp账户与原路由/每天执行/ SOXX比较，明确开发样本门缺失；不挑阈值、周期或seed。本轮没有新的收益改善，生产未改，稳定净SOXX超额未完成。

2项针对性测试通过，覆盖费用分支隔离及训练参考拒绝学习型控制器；执行notebook和NAS逐对象冷恢复验证通过。完整标签、特征和账户在Recorder，小覆盖表见label_coverage.csv。
